<a href="https://colab.research.google.com/github/patrickkirbyjr/DataPat/blob/main/wf_lag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compare Each Row to the Previous Row in SQL (Python + DuckDB)

### Scenario
Oh boy, a new email! Looks like your boss just sent over a massive table of NYC taxi trips with this message:

> "Check out this data. I need you to flag whether each trip was longer, shorter, or the same distance as the one right before it. Can you knock this out in SQL?"

In [12]:
#libraries, data, preview
import duckdb
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
df_preview = duckdb.sql(f"SELECT * FROM '{url}' LIMIT 5").df()
df_preview

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [13]:
#write query
query = f"""
WITH lagged_trips AS (
  SELECT
    VendorID,
    tpep_pickup_datetime,
    trip_distance,
    LAG(trip_distance, 1) OVER (
      PARTITION BY VendorID
      ORDER BY tpep_pickup_datetime, tpep_dropoff_datetime
    ) AS prev_trip_distance
  FROM '{url}'
  WHERE tpep_pickup_datetime >= '2023-01-01'
    AND tpep_pickup_datetime < '2023-01-02'
    AND trip_distance > 0
)
SELECT
  *,
  ROUND(trip_distance - prev_trip_distance, 2) AS distance_diff,
  CASE
    WHEN prev_trip_distance IS NULL THEN 'First Trip'
    WHEN trip_distance > prev_trip_distance THEN 'Longer'
    WHEN trip_distance < prev_trip_distance THEN 'Shorter'
    ELSE 'Same'
  END AS comparison
FROM lagged_trips
ORDER BY VendorID, tpep_pickup_datetime
LIMIT 15
"""
df = duckdb.sql(query).df()
df

,VendorID,tpep_pickup_datetime,trip_distance,prev_trip_distance,distance_diff,comparison
0,1,2023-01-01 00:00:18,2.1,NaN,NaN,First Trip
1,1,2023-01-01 00:01:08,2.1,2.1,0.0,Same
2,1,2023-01-01 00:01:34,1.6,2.1,-0.5,Shorter
3,1,2023-01-01 00:01:39,1.4,1.6,-0.2,Shorter
4,1,2023-01-01 00:01:52,1.9,1.4,0.5,Longer
5,1,2023-01-01 00:01:54,1.4,1.9,-0.5,Shorter
6,1,2023-01-01 00:02:29,20.2,1.4,18.8,Longer
7,1,2023-01-01 00:02:41,0.5,20.2,-19.7,Shorter
8,1,2023-01-01 00:02:42,1.3,0.5,0.8,Longer
9,1,2023-01-01 00:02:43,0.9,1.3,-0.4,Shorter
